# Investigación auxiliar: GUIDs con múltiples PIDs

Notebook exploratorio para investigar la anomalía detectada en el Paso 5 del análisis principal:
¿por qué hay más combinaciones GUID-PID que GUIDs únicos?

**Secciones:**
1. **Pasos 1–3** (run-01, par `ProcessGuid`/`ProcessId`): identificación del GUID centinela
   `00000000-0000-0000-0000-000000000000` como única fuente de la anomalía.
2. **Análisis unificado**: extiende la investigación a los 4 pares GUID/PID del schema
   (`ProcessGuid`, `ParentProcessGuid`, `SourceProcessGUID`, `TargetProcessGUID`).
3. **Cross-run**: verifica el patrón en los 38 runs con CSV procesado, para todos los pares.

In [ ]:
import pandas as pd

TARGET_PATH = "../dataset/run-01-apt-1/02_sysmon-run-01.csv"

df = pd.read_csv(
    TARGET_PATH,
    low_memory=False,
    dtype={
        'ProcessId': 'Int64', 'SourcePort': 'Int64', 'DestinationPort': 'Int64',
        'SourceProcessId': 'Int64', 'ParentProcessId': 'Int64',
        'SourceThreadId': 'Int64', 'TargetProcessId': 'Int64',
        'ProcessGuid': 'string', 'SourceProcessGUID': 'string',
        'TargetProcessGUID': 'string', 'ParentProcessGuid': 'string',
        'Computer': 'category', 'Protocol': 'category', 'EventType': 'category'
    }
)

print(f"Dataset cargado: {len(df):,} filas x {len(df.columns)} columnas")

In [ ]:
# Paso 1: identificar los GUIDs con más de un PID

valid = df[df['ProcessGuid'].notna() & df['ProcessId'].notna()]

# Por cada GUID, contar cuántos PIDs distintos tiene
pids_per_guid = valid.groupby('ProcessGuid')['ProcessId'].nunique()
multi_pid_guids = pids_per_guid[pids_per_guid > 1]

print(f"GUIDs con más de un PID: {len(multi_pid_guids)}")
print()
print(multi_pid_guids.sort_values(ascending=False))

In [ ]:
# Paso 2: para cada GUID conflictivo, mostrar qué PIDs tiene y en qué EventIDs aparece

for guid in multi_pid_guids.index:
    subset = valid[valid['ProcessGuid'] == guid][['EventID', 'ProcessId', 'Computer', 'Image']]
    pids = subset['ProcessId'].unique()
    eids = subset['EventID'].unique()
    print(f"GUID: {guid}")
    print(f"  PIDs: {sorted(pids)}")
    print(f"  EventIDs: {sorted(eids)}")
    print(subset.drop_duplicates(subset=['EventID', 'ProcessId']).to_string(index=False))
    print()

In [ ]:
# Paso 3: distribución de EventIDs en los registros conflictivos

conflicting_rows = valid[valid['ProcessGuid'].isin(multi_pid_guids.index)]
print(f"Total filas involucradas: {len(conflicting_rows)}")
print()
print("Distribución por EventID:")
print(conflicting_rows['EventID'].value_counts().sort_index())

---
## Análisis unificado: los 4 pares GUID/PID

Extiende la investigación anterior a todos los pares GUID/PID del schema:
`ProcessGuid/ProcessId`, `ParentProcessGuid/ParentProcessId`,
`SourceProcessGUID/SourceProcessId`, `TargetProcessGUID/TargetProcessId`.

Para cada par: cuántos GUIDs tienen más de un PID, si el GUID centinela está presente,
y si algún GUID real (no centinela) presenta la anomalía.

In [ ]:
GUID_PID_PAIRS = [
    ('ProcessGuid',       'ProcessId'),
    ('ParentProcessGuid', 'ParentProcessId'),
    ('SourceProcessGUID', 'SourceProcessId'),
    ('TargetProcessGUID', 'TargetProcessId'),
]

NULL_GUID = '00000000-0000-0000-0000-000000000000'

for guid_col, pid_col in GUID_PID_PAIRS:
    if guid_col not in df.columns or pid_col not in df.columns:
        print(f'[SKIP] {guid_col}/{pid_col}: columnas no presentes')
        continue

    valid = df[df[guid_col].notna() & df[pid_col].notna()]
    pids_per_guid = valid.groupby(guid_col)[pid_col].nunique()
    multi = pids_per_guid[pids_per_guid > 1]

    real_multi = multi[multi.index != NULL_GUID]
    has_null   = NULL_GUID in multi.index

    print('=' * 60)
    print(f'Par: {guid_col} / {pid_col}')
    print(f'  GUIDs únicos:             {valid[guid_col].nunique():,}')
    print(f'  GUIDs con >1 PID:         {len(multi)}')

    if has_null:
        null_rows = valid[valid[guid_col] == NULL_GUID]
        eids = sorted(null_rows['EventID'].unique().tolist())
        print(f'  — GUID centinela: SÍ  ({len(null_rows)} filas, '
              f'{int(multi[NULL_GUID])} PIDs distintos, EIDs: {eids})')
    else:
        print('  — GUID centinela: no')

    if len(real_multi) > 0:
        print(f'  — GUIDs reales multi-PID: {len(real_multi)}')
        for guid, n_pids in real_multi.sort_values(ascending=False).items():
            subset = valid[valid[guid_col] == guid]
            eids = sorted(subset['EventID'].unique().tolist())
            print(f'      GUID: {guid}  PIDs: {n_pids}  EIDs: {eids}')
    else:
        print(f'  — GUIDs reales multi-PID: 0  ✅')
    print()


---
## Análisis sobre todos los APT runs

Verifica si el GUID nulo (`00000000-0000-0000-0000-000000000000`) es el único conflictivo
en todos los runs, y qué EventIDs lo reciben en cada uno.

Solo se procesan los CSVs con prefijo `02_` (pipeline actualizado). Los runs sin ese archivo se saltan.

In [ ]:
import os
import glob
import pandas as pd

DATASET_ROOT = "/home/researcher/Research/phd-thesis/curso2026/dataset"
NULL_GUID    = "00000000-0000-0000-0000-000000000000"

GUID_PID_PAIRS = [
    ('ProcessGuid',       'ProcessId'),
    ('ParentProcessGuid', 'ParentProcessId'),
    ('SourceProcessGUID', 'SourceProcessId'),
    ('TargetProcessGUID', 'TargetProcessId'),
]

COLS = ['EventID',
        'ProcessGuid',       'ProcessId',
        'ParentProcessGuid', 'ParentProcessId',
        'SourceProcessGUID', 'SourceProcessId',
        'TargetProcessGUID', 'TargetProcessId']
DTYPES = {
    'ProcessId':       'Int64', 'ProcessGuid':       'string',
    'ParentProcessId': 'Int64', 'ParentProcessGuid': 'string',
    'SourceProcessId': 'Int64', 'SourceProcessGUID': 'string',
    'TargetProcessId': 'Int64', 'TargetProcessGUID': 'string',
}

summary_rows = []

run_dirs = sorted(glob.glob(os.path.join(DATASET_ROOT, "run-*")))

for run_dir in run_dirs:
    run_name = os.path.basename(run_dir)
    csv_files = glob.glob(os.path.join(run_dir, "02_sysmon-run-*.csv"))

    if not csv_files:
        print(f"[SKIP] {run_name}: no 02_sysmon CSV found")
        continue

    csv_path = csv_files[0]

    try:
        df_run = pd.read_csv(csv_path, low_memory=False,
                             usecols=COLS, dtype=DTYPES)

        row = {'run': run_name, 'total_rows': len(df_run)}
        pair_summaries = []

        for guid_col, pid_col in GUID_PID_PAIRS:
            valid = df_run[df_run[guid_col].notna() & df_run[pid_col].notna()]
            pids_per_guid = valid.groupby(guid_col)[pid_col].nunique()
            multi = pids_per_guid[pids_per_guid > 1]

            has_null  = NULL_GUID in multi.index
            real_multi = multi[multi.index != NULL_GUID]

            null_rows = int((valid[guid_col] == NULL_GUID).sum()) if has_null else 0
            null_pids = int(multi.get(NULL_GUID, 0))
            eids_null = sorted(
                valid[valid[guid_col] == NULL_GUID]['EventID'].unique().tolist()
            ) if has_null else []

            short = guid_col.replace('ProcessGuid', 'PG').replace('ProcessGUID', 'PGUID')
            row[f'{short}_null']       = has_null
            row[f'{short}_null_rows']  = null_rows
            row[f'{short}_null_pids']  = null_pids
            row[f'{short}_null_eids']  = str(eids_null)
            row[f'{short}_real_multi'] = len(real_multi)

            pair_summaries.append(
                f"{guid_col}: null={'Y' if has_null else 'n'}({null_rows}r,{null_pids}p) "
                f"real_multi={len(real_multi)}"
            )

        summary_rows.append(row)
        print(f"[OK] {run_name} ({len(df_run):,} rows)")
        for s in pair_summaries:
            print(f"       {s}")

    except Exception as e:
        print(f"[ERROR] {run_name}: {e}")
        summary_rows.append({'run': run_name, 'error': str(e)})

print("\nFINISHED")


In [6]:
summary_df = pd.DataFrame(summary_rows)

# Columnas base + una columna 'null' por par
null_cols = [
    ('PG_null',        'PG_null_rows',  'PG_null_pids',  'PG_null_eids',  'PG_real_multi',       'ProcessGuid'),
    ('Parent_null',    'Parent_null_rows', 'Parent_null_pids', 'Parent_null_eids', 'Parent_real_multi', 'ParentProcessGuid'),
    ('Source_null',    'Source_null_rows', 'Source_null_pids', 'Source_null_eids', 'Source_real_multi', 'SourceProcessGUID'),
    ('Target_PGUID_null', 'Target_PGUID_null_rows', 'Target_PGUID_null_pids', 'Target_PGUID_null_eids', 'Target_PGUID_real_multi', 'TargetProcessGUID'),
]

print(f"{'run':<20} {'rows':>8}  "
      f"{'ProcessGuid':^28}  {'ParentProcessGuid':^28}  "
      f"{'SourceProcessGUID':^28}  {'TargetProcessGUID':^28}")
print(f"{'':20} {'':8}  "
      f"{'null? rows pids real_multi':^28}  "*4)
print('-'*140)

for _, r in summary_df.iterrows():
    if 'error' in r:
        print(f"{r['run']:<20} ERROR: {r['error']}")
        continue
    cols_data = []
    for null_col, rows_col, pids_col, eids_col, real_col, label in null_cols:
        n = 'Y' if r.get(null_col, False) else 'n'
        nr = int(r.get(rows_col, 0))
        np_ = int(r.get(pids_col, 0))
        rm = int(r.get(real_col, 0))
        cols_data.append(f"{n} r={nr:<4} p={np_:<3} rm={rm}")
    print(f"{r['run']:<20} {int(r['total_rows']):>8}  "
          + '  '.join(f"{c:^28}" for c in cols_data))

print()
print("--- GUIDs reales con >1 PID (rm > 0) ---")
any_real = False
for _, r in summary_df.iterrows():
    if 'error' in r:
        continue
    for null_col, rows_col, pids_col, eids_col, real_col, label in null_cols:
        rm = int(r.get(real_col, 0))
        if rm > 0:
            print(f"  {r['run']} / {label}: {rm} GUID(s) reales con >1 PID")
            any_real = True
if not any_real:
    print("  Ninguno — el GUID centinela es el único caso en todos los pares y runs.")


         run  total_rows  multi_pid_guids  has_null_guid  null_guid_pids  null_guid_rows       null_guid_eids  other_guids_count
run-01-apt-1      363657                1           True              14              36           [3, 7, 13]                  0
run-02-apt-1      570078                1           True              51             208    [3, 7, 9, 12, 13]                  0
run-03-apt-1      196409                1           True              22              65               [3, 7]                  0
run-04-apt-1       65265                1           True               9              23                  [3]                  0
run-05-apt-1     1266782                1           True              63             284       [3, 7, 12, 13]                  0
run-06-apt-1     1574322                1           True              61             244    [3, 7, 9, 12, 13]                  0
run-07-apt-1      656322                1           True              47             141    [3, 7